# 🏁 Hackathon — AI Delivery Planning & Jira Agent

Notebook này **chạy trọn dự án Hackathon từ đầu đến cuối** (BRD/Excel → plan Jira đã duyệt) và ở mỗi bước dùng đúng công nghệ của các tuần học:

| Tuần | Công nghệ | Dùng ở đâu trong dự án |
|---|---|---|
| 1 | Python, Pydantic, **OpenAI API** (chat + structured output) | Model dữ liệu, đọc rule bằng AI, trả lời của agent |
| 2 | **Function calling · batching · retry**, Llama 3 (đổi `LLM_BASE_URL`/`LLM_MODEL`) | Registry 16 tool, 40 task/batch, `with_retry` |
| 3 | **Hugging Face** (sentence-transformers, TTS) + **Embeddings** | Embedding thật 384‑d chạy local, đọc to tóm tắt plan |
| 4 | **Pinecone** + **LangChain** | `VectorStore(backend="pinecone")`, tool → `StructuredTool`, `ProjectRetriever` |
| 5 | **RAG** + **LangGraph** | Hỏi đáp BRD có trích dẫn, agent là `StateGraph` có chờ người duyệt (`graph.py`) |

Mọi thứ trong notebook dùng **chính các module của app** (`vectors.py`, `pinecone_store.py`, `lc.py`, `graph.py`, `speech.py`, `jira_tools.py`), nên app Streamlit chạy cùng công nghệ.

Luồng trong notebook: `1 Setup → 2 Hạ tầng gọi model → 3 Vector (Chroma + Pinecone) → 4 Ingest & Plan → 5 Trùng theo nghĩa → 6 RAG → 7 Agent LangGraph → 8 Jira MCP → 9 Duyệt → 10 Sync Excel → 11 TTS → 12 Test`.

**Mặc định chạy LIVE**: cần `LLM_API_KEY` trong `.env` (STU gateway). Chat, function calling, đọc rule (structured output), trích xuất tài liệu và embedding `text-embedding-3-small` đều gọi gateway thật; cell 1 kiểm tra từng dịch vụ trước khi chạy. Hugging Face (embedding đa ngôn ngữ + TTS) chạy local. Pinecone thật khi có `PINECONE_API_KEY`, nếu không dùng index in-memory cùng giao diện. Đặt `REQUIRE_LIVE = False` để chạy bản offline (model giả lập) khi không có key. Cài thư viện: `uv pip install -r requirements.txt`.

## 1. Setup

In [ ]:
import os, sys, re, json, time, math, tempfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "tests"))       # FakeAssessor của bộ test: thay AI đọc rule khi chạy offline

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
REQUIRE_LIVE = True                                                 # True: bắt buộc gọi dịch vụ thật; False: cho phép chạy offline
LIVE = bool(os.getenv("LLM_API_KEY") or os.getenv("API_KEY"))
if REQUIRE_LIVE and not LIVE:
    raise RuntimeError("Thiếu LLM_API_KEY trong .env. Điền key, hoặc đặt REQUIRE_LIVE = False để chạy offline.")
BRIDGE = None                                                       # kết nối Jira MCP (mở ở cell kiểm tra bên dưới)
WORK = Path(tempfile.mkdtemp(prefix="hackathon_nb_"))
print("project root:", ROOT, "\nscratch      :", WORK, "\nLIVE LLM     :", LIVE)

In [ ]:
# Kiểm tra key dùng được cho từng phần (không in key)
from delivery_agent import llm
def check(name, fn):
    try:
        print(f"✅ {name:34s}", str(fn())[:60].replace(chr(10), " ")); return True
    except Exception as e:
        print(f"❌ {name:34s}", type(e).__name__, str(e)[:100]); return False

if LIVE:
    _c = llm.get_client()
    _schema = {"type": "object", "properties": {"category": {"type": "string", "enum": ["BE", "FE"]}}, "required": ["category"], "additionalProperties": False}
    ok = [check("chat.completions", lambda: _c.chat.completions.create(model=llm.chat_model(), messages=[{"role": "user", "content": "say ok"}]).choices[0].message.content),
          check("structured output (json_schema)", lambda: llm.structured_completion(_c, system="classify", user="Create REST API", name="c", schema=_schema, what="check").choices[0].message.content),
          check("embeddings", lambda: f"dim={len(_c.embeddings.create(model=llm.DEFAULT_EMBEDDING_MODEL, input=['hi']).data[0].embedding)}")]
    print("Pinecone:", "có PINECONE_API_KEY" if os.getenv("PINECONE_API_KEY") else "không có key -> dùng index in-memory")
    if os.getenv("JIRA_MCP_COMMAND") or os.getenv("JIRA_MCP_URL"):
        from delivery_agent.jira_tools import JiraBridge
        from delivery_agent.config import load_config as _lc
        def _open_jira():
            global BRIDGE
            BRIDGE = JiraBridge(_lc()).start()
            return f"{len(BRIDGE.tools)} tool"
        ok.append(check("Jira MCP server", _open_jira))
    else:
        print("Jira MCP: chưa cấu hình JIRA_MCP_COMMAND -> bỏ qua section 8")
    assert all(ok), "Có dịch vụ không dùng được - kiểm tra .env"

## 2. Hạ tầng gọi model (Tuần 1–3)

- **Pydantic** (`models.py`): `Task` dùng cho validate, JSON và sinh schema function calling.
- **OpenAI SDK → STU gateway** (`llm.get_client`): đổi `LLM_BASE_URL`/`LLM_MODEL` là chạy được Llama 3 hoặc model khác.
- **Retry** (`with_retry`): backoff + jitter chỉ cho 429/timeout/connection/5xx; 4xx fail ngay.
- **Embedder**: batch 100 text/request + cache theo hash. Provider `hf` = **Hugging Face sentence-transformers** chạy local.

In [ ]:
import httpx, openai
from delivery_agent.llm import Embedder, with_retry
from delivery_agent.models import Task, CATEGORIES
from delivery_agent.db import Repository

REPO = Repository(f"sqlite:///{(WORK / 'app.db').as_posix()}")        # SQLAlchemy: runs, tasks, plans, approvals, audit, cache
HF_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"      # Hugging Face đa ngôn ngữ (có tiếng Việt), chạy local
EMB = (Embedder(provider="openai", cache=REPO.cache("embedding")) if LIVE                        # text-embedding-3-small qua STU gateway
       else Embedder(provider="hf", model=HF_MODEL, cache=REPO.cache("embedding")))              # offline: Hugging Face
HF_EMB = Embedder(provider="hf", model=HF_MODEL)                                                  # luôn có sẵn để so sánh

v = EMB.embed(["Create Customer API", "Build REST endpoint for customers", "Order pizza"])
cos = lambda a, b: sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))
print(f"{EMB.provider}:{EMB.model} dim={len(v[0])} | API~endpoint={cos(v[0], v[1]):.2f}  API~pizza={cos(v[0], v[2]):.2f}")
h = HF_EMB.embed(["Create Customer API", "Build REST endpoint for customers", "Order pizza"])
print(f"hf:{HF_MODEL.split('/')[-1]} dim={len(h[0])} | API~endpoint={cos(h[0], h[1]):.2f}  API~pizza={cos(h[0], h[2]):.2f}")

# Retry: 2 lần 429 rồi thành công; 400 thì raise ngay
req = httpx.Request("POST", "https://x")
err = lambda cls, code: cls("boom", response=httpx.Response(code, request=req), body=None)
n = {"i": 0}
def flaky():
    n["i"] += 1
    if n["i"] < 3: raise err(openai.RateLimitError, 429)
    return "OK"
log = []
print(with_retry(flaky, what="demo", on_retry=log.append, wait_max=0.05), "sau", n["i"], "lần |", log[0])
try: with_retry(lambda: (_ for _ in ()).throw(err(openai.BadRequestError, 400)), wait_max=0.05)
except openai.BadRequestError: print("400 -> không retry ✔")

## 3. Vector layer: ChromaDB và Pinecone (Tuần 3–4)

Dự án lưu **3 collection** (`tasks`, `rules`, `docs`, mọi record có `project_key`) trong **ChromaDB** (mặc định) hoặc **Pinecone** (`VECTOR_BACKEND=pinecone`, `pinecone_store.py`): cùng `VectorStore`, mỗi collection là một *namespace* của một index, `project_key` là filter. Có `PINECONE_API_KEY` → Pinecone thật (serverless), không thì index in-memory cùng giao diện.

In [ ]:
import chromadb
from types import SimpleNamespace
from delivery_agent.vectors import VectorStore, task_text, chunk_markdown

STORE = VectorStore(EMB, path=WORK / "chroma")          # Chroma persistent, dùng embedder Hugging Face

from delivery_agent.pinecone_store import InMemoryPineconeIndex

# Cùng VectorStore, backend Pinecone: một index, mỗi collection (tasks/rules/docs) một namespace
if os.getenv("PINECONE_API_KEY"):
    STORE_PC, PC_KIND = VectorStore(EMB, backend="pinecone"), "Pinecone THẬT"
else:
    STORE_PC, PC_KIND = VectorStore(EMB, backend="pinecone", pinecone_index=InMemoryPineconeIndex()), "Pinecone in-memory (không có PINECONE_API_KEY)"
print("Chroma:", STORE.stats(), "| Pinecone:", PC_KIND, "| backend:", STORE_PC.backend)

## 4. Ingest & lập Plan (Tuần 1–2: Pydantic, structured output, batching)

`AgentSession` là một phiên chat = một **run** trong DB. Bước: nạp BRD (được **chunk + embed + lưu Chroma** để RAG) → đọc Excel → **AI đọc rule** (structured output `enum`, batch 40 task) → **code** so sánh estimate và validate → dựng Epic/Story/Task/Subtask.

> Bước *đọc rule* gọi **AI thật** (`RuleAssessor`, structured output qua Chat Completions, cache vào DB nên chạy lại không tốn thêm). Không có key thì dùng `FakeAssessor`.

In [ ]:
from conftest import FakeAssessor
from delivery_agent.agent import AgentSession, run_turn
from delivery_agent.ai_rules import RuleAssessor
from delivery_agent.config import load_config

USE_REAL_RULE_AI = LIVE
def new_session(**kw):
    # assessor=None -> AgentSession tự tạo RuleAssessor thật (cache DB + bằng chứng task cũ + rule retrieval)
    return AgentSession(load_config(), offline=True, assessor=None if USE_REAL_RULE_AI else FakeAssessor(),
                        repo=REPO, vectors=STORE, out_root=WORK / "runs", jira_bridge=BRIDGE, **kw)

run1 = new_session()
run1.add_file("brd_customer_portal.md", (ROOT / "samples" / "brd_customer_portal.md").read_bytes())    # index BRD vào Chroma (docs)
res = run1.analyze(["task_review.xlsx"])
print("\n".join(res["summary"]))
print("\n" + run1.show_plan("tree")["tree"])
print("\nChroma sau run 1:", STORE.stats())

In [ ]:
# Estimate review: AI đọc rule -> CODE so sánh số;  Validate: code
for f in run1.show_plan("estimate_warnings")["estimate_warnings"]:
    print(f"[{f['severity']:6}] {f['task'][:34]:34} {f['rule']:16} hiện tại {f['currentEstimate'] or '-':>3} -> gợi ý {f['suggestedEstimate']}")
for i in run1.show_plan("validation_issues")["validation_issues"]:
    print(f"[{i['severity']:6}] {i['task'][:34]:34} {i['kind']}: {i['detail']}")

# Tool-call + DB: toàn bộ lịch sử đã được lưu
import sqlalchemy as sa
eng = next(getattr(REPO, a) for a in ("engine", "_engine") if hasattr(REPO, a))
print("\nbảng DB:", sa.inspect(eng).get_table_names())

## 5. Trùng lặp theo *nghĩa* với lịch sử (Tuần 3–4: Embeddings + vector DB)

Run 2 nhận một Excel diễn đạt lại task bằng từ khác. Fuzzy-match theo chữ không bắt được, **embedding Hugging Face** bắt được (scope `history`). Cùng truy vấn chạy trên backend **Pinecone** của `VectorStore` để so sánh.

In [ ]:
import openpyxl
wb = openpyxl.load_workbook(ROOT / "samples" / "task_review.xlsx"); ws = wb.active
cols = {str(c.value).strip(): c.column for c in ws[1] if c.value}
REWORD = {"Create Customer API": "Create customer REST API", "Build registration screen": "Build customer registration screen"}
for row in ws.iter_rows(min_row=2):
    c = ws.cell(row[0].row, cols["Title 2"])
    if c.value in REWORD: c.value = REWORD[c.value]
reworded = WORK / "task_review_v2.xlsx"; wb.save(reworded)

run2 = new_session()
run2.config["validation"]["semantic_threshold"] = 0.75     # mặc định 0.88; hạ để bắt cả diễn đạt lại
run2.analyze([str(reworded)])
for i in run2.show_plan("validation_issues")["validation_issues"]:
    if i["kind"] == "Possible Duplicate": print(f"[{i['severity']}] {i['task']}  ->  {i['detail']}")

# Cùng câu hỏi trên backend Pinecone (Pinecone nhất quán chậm: chờ dữ liệu vừa ghi hiện ra)
for r in (run1, run2): STORE_PC.index_tasks(r.project_key, r.run_id, r.tasks)
for _ in range(15):
    hits = STORE_PC.similar_tasks(run2.project_key, ["Create customer REST API"], k=2, exclude_run=run2.run_id)[0]
    if hits: break
    time.sleep(2)
for h in hits:
    print(f"Pinecone  {h.score:.2f}  {h.meta['title']}  ({h.meta['category']}, {h.meta['estimate']})")

## 6. RAG trên BRD với LangChain (Tuần 4–5)

**Index** (đã làm ở bước 4: chunk theo heading + overlap → embedding → vector store `docs`) → **Retrieve** → **Generate có trích dẫn**. `ProjectRetriever` (`lc.py`) là **LangChain `BaseRetriever`** bọc vector store của dự án; chuỗi LCEL `retriever | format | prompt | llm | parser`. Tool `ask_requirements` của app đi qua đúng retriever này (và `answer=true` chạy cùng chuỗi). Có key → `ChatOpenAI` trỏ STU gateway; không key → trả lời trích xuất từ chính các đoạn truy xuất được.

In [ ]:
from delivery_agent.lc import ProjectRetriever, rag_chain, default_chat_model
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter

retriever = ProjectRetriever(store=STORE, project_key=run1.project_key, run_id=run1.run_id, k=3)
chat_model = default_chat_model() if LIVE else RunnableLambda(    # offline: "LLM" trích đoạn liên quan nhất, vẫn giữ trích dẫn
    lambda pv: "(offline) " + pv.to_string().split("Passages:")[1].split("Question:")[0].strip()[:400])
rag = rag_chain(retriever, chat_model)                              # retriever | format_docs | prompt | llm | parser
for q in ["Khách hàng đăng ký trực tuyến như thế nào?", "Dữ liệu cũ được chuyển sang đâu?"]:
    print("Q:", q, "\n  A:", rag.invoke(q), "\n")

# Tool của agent trong app dùng cùng retriever: passages kèm file / chunk / relevance
for p in run1.ask_requirements("Khách hàng đăng ký như thế nào?", k=2)["passages"]:
    print(f"[{p['file']}#{p['chunk']}] relevance={p['relevance']}")

# So sánh với splitter của LangChain
print("chunk_markdown của dự án:", len(chunk_markdown((ROOT/'samples'/'brd_customer_portal.md').read_text(encoding='utf-8'), 400, 50)),
      "| RecursiveCharacterTextSplitter:", len(RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50).split_text((ROOT/'samples'/'brd_customer_portal.md').read_text(encoding='utf-8'))))

## 7. Agent: function calling + LangChain tools + LangGraph (Tuần 2, 4, 5)

- Registry **16 tool** (Pydantic → JSON schema, validate args, lỗi quay lại cho model) → `langchain_tools()` chuyển thành LangChain **`StructuredTool`** (`lc.py`).
- `AgentGraph` (`graph.py`) là **`StateGraph`**: `agent → (tools → agent)* → END`, có **`MemorySaver` checkpointer** và **`interrupt_before=["tools"]`**: *người* thấy tool sắp chạy rồi mới cho chạy tiếp (human‑in‑the‑loop). App Streamlit dùng đúng lớp này (bật "✋ Duyệt từng bước").
- Ngoài 16 tool của dự án, agent có **toàn bộ tool của Jira MCP server** (`jira_*`, xem section 8). Tool đọc chạy ngay; tool ghi chỉ được *đề xuất* và chờ người duyệt. Model chat: gateway thật nếu có key, ngược lại một model kịch bản.

In [ ]:
from delivery_agent.graph import AgentGraph
from delivery_agent.lc import langchain_tools
from delivery_agent.tools import REGISTRY

assert not any(n == "push" or n.startswith("create_") for n in REGISTRY.tools)     # 16 tool riêng của dự án không ghi Jira; tool Jira MCP có cổng duyệt (section 8)

class ScriptedChat:
    # Model giả (khi không có key): gọi lần lượt các tool đã định, rồi trả lời text.
    def __init__(self, calls, final=None):
        self.final = final or ("Đã phân tích xong: 6 task, có cảnh báo estimate; BRD nêu khách tự đăng ký online; "
                               "đã tìm task tương tự ở các run trước.")
        self.calls = list(calls); self.chat = SimpleNamespace(completions=SimpleNamespace(create=self.create))
    def create(self, model, messages, tools):
        if self.calls:
            name, args = self.calls.pop(0); i = len(messages)
            tc = SimpleNamespace(id=f"call{i}", function=SimpleNamespace(name=name, arguments=json.dumps(args)),
                                 model_dump=lambda n=name, a=args, i=i: {"id": f"call{i}", "type": "function",
                                                                        "function": {"name": n, "arguments": json.dumps(a)}})
            msg = SimpleNamespace(content="", tool_calls=[tc])
        else:
            msg = SimpleNamespace(content=self.final, tool_calls=None)
        return SimpleNamespace(choices=[SimpleNamespace(message=msg)])

print("tool của dự án:", len(REGISTRY.tools), "| LangGraph: agent <-> tools, MemorySaver, interrupt_before=['tools']")

In [ ]:
agent_session = new_session()
agent_session.add_file("brd_customer_portal.md", (ROOT / "samples" / "brd_customer_portal.md").read_bytes())
client = llm.get_client() if LIVE else ScriptedChat([
    ("analyze", {"files": ["task_review.xlsx"]}),
    ("ask_requirements", {"question": "Khách hàng đăng ký như thế nào?", "k": 2}),
    ("search_similar_tasks", {"text": "Create customer REST API", "k": 2})])
lc_tools = langchain_tools(agent_session)
print("StructuredTool:", len(lc_tools), "| tham số show_plan:", list(lc_tools["show_plan"].args_schema.model_json_schema()["properties"]))

graph = AgentGraph(agent_session, client=client, interrupt=True,          # interrupt=True: dừng trước mỗi lượt chạy tool
                   on_tool=lambda n, a, r: print("   ✔", n, "-> lỗi" if "error" in r else "-> ok"))
question = ("Phân tích file mẫu task_review.xlsx (tài liệu brd_customer_portal.md đã đính kèm); tài liệu nói gì về đăng ký khách hàng? "
            "và trước đây có task nào giống 'Create customer REST API' chưa?")
result = graph.run([{"role": "user", "content": question}])
AUTO_APPROVE = True                                              # False: dừng ở mỗi tool để người quyết định
while result.paused:                                             # graph đang dừng trước node 'tools'
    for tc in result.paused:
        print("⏸  chờ duyệt tool:", tc["function"]["name"], tc["function"]["arguments"][:100])
    if not AUTO_APPROVE: break
    result = graph.resume(approve=True)                          # người duyệt -> chạy tiếp (approve=False: báo model là bị từ chối)
print("\n💬", result.text[:700])
print("\ntool đã chạy:", [(c["name"], c["ok"], f"{c['duration_ms']}ms") for c in agent_session.repo.tool_calls(agent_session.run_id)])

## 8. Jira MCP đầy đủ cho agent (Tuần 4–5: MCP + tools)

`jira_tools.py` kết nối Jira MCP server (`mcp-atlassian`) **một lần** (giữ kết nối mở), tự **khám phá toàn bộ tool** của server (63 tool: issue, search/JQL, sprint, board, link, transition, comment, worklog, version, service desk…) và đưa vào agent cùng 16 tool của dự án.

| Loại tool | Cách xử lý |
|---|---|
| **Đọc** (server đánh dấu `readOnlyHint`) | Chạy ngay |
| **Ghi** (tạo/sửa/xoá/gán/transition/comment/link/sprint…) | Agent chỉ *đề xuất* → vào hàng chờ `pending_approval` → **người duyệt** mới chạy (panel **Jira actions** trong app, hoặc `decide_jira_action`) |

Chính sách đổi bằng `jira.agent_write_policy` trong `settings.yaml`: `approve` (mặc định) · `allow` (agent được ghi thẳng) · `deny` (chỉ đọc). Tạo hàng loạt issue từ tài liệu vẫn nên đi theo luồng plan → duyệt → push.

In [ ]:
tb = agent_session.jira_toolbox
if tb is None:
    print("Chưa kết nối Jira MCP (cần JIRA_MCP_COMMAND trong .env) - bỏ qua section này")
else:
    reads = sorted(n for n in tb.names if not tb.is_write(n)); writes = sorted(n for n in tb.names if tb.is_write(n))
    print(f"{len(tb.names)} tool Jira MCP | chính sách ghi: {tb.policy} | {len(reads)} đọc, {len(writes)} ghi")
    print("\nGHI (cần duyệt):", ", ".join(writes))
    print("\nĐỌC (chạy ngay):", ", ".join(reads)[:500], "...")
    out = tb.call("jira_search", {"jql": f"project = {agent_session.project_key} ORDER BY created DESC", "limit": 5})
    print("\njira_search ->", [(i["key"], i.get("summary", "")[:40]) for i in out["result"].get("issues", [])])

In [ ]:
def chat(session, client, question, auto_approve=True):
    # Một lượt hỏi agent qua LangGraph; mỗi tool dừng chờ người cho chạy (auto_approve=True: tự cho chạy tool).
    g = AgentGraph(session, client=client, interrupt=True)
    result = g.run([{"role": "user", "content": question}])
    while result.paused and auto_approve:
        for tc in result.paused:
            print("⏸  tool:", tc["function"]["name"], tc["function"]["arguments"][:110])
        result = g.resume(approve=True)
    return result.text

if tb is not None:
    jira_client = llm.get_client() if LIVE else ScriptedChat(
        [("jira_search", {"jql": "project = KAN AND issuetype = Epic", "limit": 5}),
         ("jira_add_comment", {"issue_key": "KAN-22", "body": "Kiểm tra từ agent"})],
        final="Đã liệt kê Epic; comment vào KAN-22 đang chờ duyệt.")
    print("💬", chat(agent_session, jira_client,
                     "Trên Jira project KAN có những Epic nào? Sau đó thêm comment 'Kiểm tra từ agent' vào KAN-22.")[:600])
    print("\n⏳ Hàng chờ duyệt:", [(a["id"], a["tool"], a["args"]) for a in agent_session.pending_jira_actions()])

In [ ]:
# Người duyệt. Mặc định KHÔNG ghi lên Jira thật; đổi APPROVE_JIRA_WRITE = True để duyệt các thao tác đang chờ.
APPROVE_JIRA_WRITE = False
if tb is not None:
    for a in agent_session.pending_jira_actions():
        if APPROVE_JIRA_WRITE:
            print("✅ duyệt", a["tool"], "->", agent_session.decide_jira_action(a["id"], True, "Lan (PM)").get("status"))
        else:
            print("⏳ chưa duyệt", a["tool"], a["args"], "(đặt APPROVE_JIRA_WRITE = True để thực thi)")

## 9. Duyệt bởi con người (approval gắn SHA-256)

Plan chỉ được push khi có approval **ràng buộc với hash của plan**; sửa plan sau khi duyệt → phải duyệt lại. Estimate gợi ý được áp dụng thành **phiên bản plan mới**.

In [ ]:
from delivery_agent.approval import plan_hash, record_decision, require_approval, ApprovalError
plan_session = run1
v_before = plan_session.plan["version"]
plan_session.apply_suggested_estimates(only_missing=True)
print(f"plan v{v_before} -> v{plan_session.plan['version']} (áp estimate gợi ý)")

record_decision(plan_session.plan_path, plan_session.plan, "Lan (PM)", True, plan_session.audit, on_record=plan_session.record_approval)
print("đã duyệt, hash", plan_hash(plan_session.plan)[:12], "->", require_approval(plan_session.plan_path, plan_session.plan)["decision"])
plan_session.update_tasks([{"task_id": plan_session.tasks[0].id, "field": "estimate", "value": "99h"}])
try: require_approval(plan_session.plan_path, plan_session.plan)
except ApprovalError as e: print("sửa sau khi duyệt ->", e)

## 10. Đồng bộ Excel → Task (embedding thật, update thay vì tạo mới)

Ghép dòng theo thứ tự **ID thật → title chuẩn hoá → *cùng nghĩa (embedding HF, ngưỡng `SEMANTIC_MATCH`)* → fuzzy**. Task đã có Jira key được **update**, không tạo trùng; task bị xoá chỉ rời khỏi plan, **không xoá trên Jira**.

In [ ]:
sync = new_session()
sync.analyze(["task_review.xlsx"])
t0 = sync.tasks[0]
ref = next(n["ref"] for n in sync.plan["planned_issues"] if n["task_id"] == t0.id)
sync.after_push({"created": {ref: "KAN-7"}, "linked": [], "skipped": {}})       # giả lập đã push -> có Jira key
print("task", t0.title, "đã có Jira key", sync.tasks[0].jira_key)

wb = openpyxl.load_workbook(ROOT / "samples" / "task_review.xlsx"); ws = wb.active
cols = {str(c.value).strip(): c.column for c in ws[1] if c.value}
for row in ws.iter_rows(min_row=2):
    r = row[0].row
    if ws.cell(r, cols["Title 2"]).value == t0.title:
        ws.cell(r, cols["Original Estimate"]).value = 99                         # đổi estimate
    if ws.cell(r, cols["Title 2"]).value == "Build registration screen":
        ws.cell(r, cols["Title 2"]).value = "Build customer registration screen"                 # đổi tên + xoá ID -> phải ghép theo NGHĨA
        ws.cell(r, cols["ID"]).value = None
n = ws.max_row + 1
for h, val in {"Work Item Type": "Task", "ID": 2001, "Parent": 1001, "Title 2": "Set up CI pipeline", "Original Estimate": 8}.items():
    ws.cell(n, cols[h]).value = val
edited = WORK / "edited.xlsx"; wb.save(edited)

import delivery_agent.sync as sync_mod
sync_mod.SEMANTIC_MATCH = 0.75         # mặc định 0.90; cặp diễn đạt lại này chỉ đạt ~0.77–0.83 (tuỳ model embedding)
rep = sync.sync_excel(str(edited), apply=False)
print(rep["summary"])
print("ghép theo:", {k: v for k, v in rep["matched_by"].items() if v != "id"} or rep["matched_by"])
for c in rep["changed"]: print("  changed:", c["title"], "|", c["field"], c["old"], "->", c["new"])
print("  added  :", rep["added"], "| removed:", rep["removed"])

In [ ]:
rep = sync.sync_excel(str(edited), apply=True)
node = next(x for x in sync.plan["planned_issues"] if x["task_id"] == t0.id)
print("applied:", rep["applied"], "| plan v", sync.plan["version"], "| task đã push ->", {k: node[k] for k in ("existing_key", "update_fields")}, "(UPDATE, không tạo mới)")

## 11. Đọc to bản tóm tắt plan bằng Hugging Face Text-to-Speech (Tuần 3)

`session.speak_plan()` (tool `read_plan_aloud` và nút 🔊 trong app, `speech.py`) đọc tóm tắt plan bằng `transformers.pipeline("text-to-speech")`: model `facebook/mms-tts-vie` (tiếng Việt, số được đọc thành chữ), chạy local; đổi `TTS_MODEL` để dùng model khác (vd `facebook/mms-tts-eng`).

In [ ]:
from IPython.display import Audio, display

info = plan_session.speak_plan()
print(info["text"], "\n->", plan_session.downloads["plan_summary.wav"], f"({info['seconds']}s, {info['model']})")
display(Audio(filename=str(plan_session.downloads["plan_summary.wav"])))

## 12. Kiểm thử toàn dự án
Bộ test chạy không cần mạng (SQLite in-memory, Chroma tạm, embedder giả) và là tài liệu sống về hành vi mong đợi.

In [ ]:
import subprocess
py = ROOT / ".venv" / "Scripts" / "python.exe"
r = subprocess.run([str(py if py.exists() else sys.executable), "-m", "pytest", "-q", "--no-header", "-p", "no:cacheprovider"],
                   cwd=ROOT, capture_output=True, text=True, timeout=900)
print((r.stdout or r.stderr)[-600:])

## 13. Ôn nhanh để bảo vệ Hackathon

| Câu hỏi | Trả lời ngắn | Bằng chứng |
|---|---|---|
| Agent dùng Jira MCP thế nào mà vẫn an toàn? | Có đủ 63 tool; đọc chạy ngay, **ghi chỉ đề xuất và chờ người duyệt** (policy `approve`) nên prompt injection trong BRD không tự ghi được | `jira_tools.py`, cell 8 |
| Function calling + structured output khác gì? | Cái đầu để model *chọn hành động*, cái sau ép *hình dạng đáp án* (`enum` category) | `tools.py`, `ai_rules.OUTPUT_SCHEMA` |
| Vì sao AI đọc rule còn code so sánh số? | LLM không tự bịa ngưỡng; so sánh `>` phải deterministic | `ai_rules.assess` |
| Retry/Batching ở đâu? | `with_retry` (429/5xx), 40 task/batch × 4 song song, embedding 100/request + cache hash | `llm.py`, `ai_rules.py` |
| Vì sao embedding HF và OpenAI không trộn? | Vector khác không gian → collection đặt tên theo model | `VectorStore.__init__` |
| Chroma hay Pinecone? | Chroma: local/dev; Pinecone: managed, scale, namespace. Giao diện upsert/query/filter tương đương | cell 3 + 5 |
| RAG hay fine-tune cho BRD? | RAG: tài liệu đổi liên tục, có trích dẫn, rẻ | cell 6 |
| LangGraph thêm gì cho vòng lặp tay? | State rõ ràng, checkpointer, `interrupt_before` cho human-in-the-loop | cell 7 |
| Scale 10.000 task? | batch + song song, cache DB, retrieve rule thay vì gửi cả file, PostgreSQL | `ai_rules.py`, `db.py` |
| Approval chống gì? | Sửa plan sau khi duyệt (hash SHA-256) | `approval.py`, cell 8 |
| Embedding hỏng thì sao? | Plan vẫn chạy (`_vec` nuốt lỗi, sidebar 🟠) | `agent._vec` |

Tài liệu liên quan: `README.md`, `docs/BUSINESS_DESCRIPTION.md`, `docs/HUONG_DAN_THUYET_TRINH.md`, `docs/TEST_CASES.md`.